## DS256 - Scalable Systems for Data Science | Jan 2025
# Assignment 1 : Spark Dataframes

## Posted on: 2025-03-24
## Deadline: 2025-04-13 11:59PM IST
## Maximum Points: 150

### Changelog:
----
* v0: Initial release of questions. [RM, RS]
* v1: Added Q3.2. Modified output format of Q3.1. Modified evaluation cell and auto grading comments. [RM]

### Common Instructions
----
* You must ONLY edit cells and regions within those cells that allow changes. **DO NOT MODIFY other cells**. This can cause the evaluation script to break and you **WILL** be penalized or get **ZERO** points.
* You MUST NOT use the string **###!@** anywhere in your code or comments. We will be using this special substring for pattern matching during evaluations.
* You may declare **all** your valid imports and user-defined functions in the cells that are provided. Otherwise, all parts of your answer must remain between the space provided for the solution to each question.
* You must only use transformations and actions over Spark DataFrames and Spark SQL to solve the assignment (**Spark Core**). You **MUST NOT** use Spark RDDs, MLLib, etc, unless specified.
 <!-- * https://spark.apache.org/docs/latest/api/python/reference/pyspark.html#rdd-apis -->
* Most of your processing to solve the problem should be done **within Spark**. Minimal post processing may be done in the Python driver code.
* You **must not** use Numpy, Scipy, etc. within the **driver** code. But you may use standard Python libraries as part of lambda expressions or functions passed to Spark transformations and actions.
<!-- * You must not use the default statistics operation (RDD.stats()) available in the **Spark Numeric RDD**. -->
* Our evaluations will include **alternate input test cases** dataset that have the same format but different contents/sizes.
* We will provide **reference outputs** for the test inputs. You can use these to verify the correctness of your solutions.
* A tangible fraction of the assessment goes towards your scalability evaluation, plots and detailed report. So complete the coding early and spend time on the experiments, analysis of performance and the report.
* **NOTE (Trigger Warning):** *Part of the assignment involves filtering out banned words. Kindly take care when processing this data as it may have sensitive words that are (by definition) not polite and potentially upsetting.*
<br>

### **IMPORTANT:** Academic Integrity
----
 The assignment must be completed by yourself without any assistance from others or from online sources, ChatGPT, Copilot, etc. Taking online help for standard API references or clearing simple doubts on Python/Spark is allowed. If **any cases of plagiarism are detected, you may get a failing grade in the course and/or be reported to the Institute for further action**. Please follow IISc’s Policy on Academic Integrity, https://iisc.ac.in/about/student-corner/academic-integrity/ .


### Submission Guidelines
----

1. Copy the **LATEST** template notebook to your local Google drive. Change the name of the notebook to **assignment1_sol.ipynb**. Proceed to complete the assignment within colab.
2. Make edits only in cells and regions that are clearly marked for modification. **Do not change the template in any other way**. We will be automatically parsing relevant cells and functions during grading. If these instructions are not followed (e.g., even an extra space in an unauthorized part), you can get a **zero** for the Assignment.
3. After you've solved the questions, verify that the output that you generate passes the **validation check** of the data types, and matches the reference outputs that are provided for the sample inputs. Note that for **floats**, only the first 3 digits of precision will be checked, but you should not make any changes to the default precision in your code. If the output data type is not valid as specified, you will get a zero for that problem.
4. Each of the problems should take no longer than 5x the time as given for our reference outputs. If it takes longer, the problem will not be evaluated.
5. When you are ready to submit the assignment, download the notebook as a **(.py)** file to your local machine. You can do so by going to **File > Download > Download (.py)** in your colab environment.
6. Place the **assigment1_sol.py** file and the PDF of the report with filename **<IISchandle>.pdf** in a folder named with your iisc mail handle **e.g., mradhika/assigment1_sol.py** and **mradhika/mradhika.pdf** if your IISc email address is mradhika@iisc.ac.in.
7. Compress the folder as a **.zip** file and upload for submission under the **Assignment 1 - Spark Programming** resource on Teams. When extracted, the .zip file should return a single folder with a python file and the report within it. Nothing more. Nothing less.

## About Common Crawl
-----
The Common Crawl dataset (https://commoncrawl.org/) is a publicly available, petabyte-scale web archive that provides freely accessible web crawl data collected monthly since 2008. It consists of raw web page data, metadata, and text extractions from billions of web pages across multiple languages and domains.

Common Crawl has been instrumental in pretraining models like GPT, BERT, and CLIP by providing vast amounts of textual data. It offers an extensive and diverse snapshot of the internet, making it a valuable resource for training large-scale AI models. Unlike proprietary datasets, it is freely available, enabling research and development without high data collection costs.

With the rise of Large Language Models (LLMs), datasets such as the Common Crawl have gained immense importance. It's been used as the primary source for training LLama.

As the data available in such datasets is vast, raw and unfiltered, preprocessing the data becomes unavoidable before use for training of any AI/LLM models.


## About this Assignment
-----
In this assignment, you will be performing several of the pre-processing steps required before model training of LLMs usng Spark. These are inspired by various articles by Meta, Falcon, etc. on such data engineering for LLMs done using Spark at massive scales. So you'll be mimicking industry's ML pipelines, albeit at a more modest scale due to limited compute resources. You'll also be bringing scientific rigor through a detailed performance evaluation of your solution.

### Report ***(30 points)***
In particular, you will examine the weak scaling behavior, resource utilization (CPU, memory, disk, network, etc.), effect of partitioning, use of different transformation/actions on the runtime, etc. You should use your knowledge of the internals of Spark and scaling of distibuted systems to explain your experimental results. These should be part of a detailed report your will enclose as a PDF.

### Output Validation
**NOTE:** When we evaluate your assignment, we will sort the output dataframe you return based on specific columns. In each question, we will mention the output column(s) we will sort on. Performing a `diff` on the sorted output dataframe MUST match our refernce outputs also present in the same sorted order. You *DO NOT* need to perform the sorting when you return the output dataframe. Our test harness will do so.

### Some Useful resources:

* RefinedWeb Dataset for Falcon LLM: https://arxiv.org/abs/2306.01116
* LLama arxiv paper: https://arxiv.org/abs/2302.13971
* A blog post on LLM Pre-processing pipeline: https://blog.christianperone.com/2023/06/appreciating-llms-data-pipelines/
* A blog post on using Spark for LLM training at Meta: https://engineering.fb.com/2017/02/07/core-infra/using-apache-spark-for-large-scale-language-model-training/


### Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Load necessary files and folders

In [3]:
!mkdir -p /content/data
!ln -s /content/drive/Shareddrives/ds256-2025-public/input/cc_small.parquet /content/data/

In [4]:
!ln -s /content/drive/Shareddrives/ds256-2025-public/input/blacklists /content/data/

In [5]:
!ln -s /content/drive/Shareddrives/ds256-2025-public/input/banned_words.txt /content/data/

In [6]:
!pip install pyspark

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

### Create Spark Session

In [8]:
spark = SparkSession.builder.appName("ssds_assignment_1") \
  .config("spark.executor.memory", "10g") \
  .config("spark.driver.memory", "10g") \
  .config("spark.memory.offHeap.enabled", "true") \
  .config("spark.memory.offHeap.size", "10g") \
  .getOrCreate()

In [9]:
spark.sparkContext.defaultParallelism

2

## About the Given Data
-----

1. We have created a subset of Common Crawl data, saving only the relevant fields in a parquet file that is loaded into the `input_df` dataframe.


In [10]:
# NOTE: This code snippet will be replaced during evaluation with the relevant sample input files.
input_df = spark.read.parquet("/content/data/cc_small.parquet")
input_df.show()

+----------+--------------------+--------------------+-----------+--------------------+--------------+-------------------------------------+
|  rec_type|                 url|                date|   language|                type|content_length|                              content|
+----------+--------------------+--------------------+-----------+--------------------+--------------+-------------------------------------+
|  warcinfo|                NULL|2024-10-16T10:38:17Z|       NULL|application/warc-...|           372|                 Software-Info: ia...|
|conversion|http://003461.cn/...|2024-10-03T09:47:25Z|    zho,jpn|          text/plain|        157267| 人妻在线高清_小知識分享：密封材料...|
|conversion|http://0832kyj.co...|2024-10-03T10:01:17Z|    zho,eng|          text/plain|          1095|  产品租赁--内江市市中区科惠锐五金...|
|conversion|http://0a.acceler...|2024-10-03T11:19:34Z|    zho,eng|          text/plain|         19014|    DC酒吧-目的和使命-欧洲杯投注赔...|
|conversion|http://0nq.bxnxb....|2024-10-03T10:59:33Z|   

2. We have also loaded and created a dataframe `blacklisted_urls_df` for all blacklisted URLs from from https://dsi.ut-capitole.fr/blacklists/ along with the `category ` they fall under, and another dataframe `banned_words_df` with Banned English Words, taken from  https://github.com/Hesham-Elbadawi/list-of-banned-words/tree/master *(view with caution)*. These dataframes are passed as input to the relevant question. *(Do not use the files directly)*

In [11]:
blacklisted_urls_df = spark.read.text("/content/data/blacklists/*/domains")
blacklisted_urls_df = blacklisted_urls_df.withColumn("path", F.input_file_name())
blacklisted_urls_df = blacklisted_urls_df.withColumn("blacklisted_url", F.col("value")).withColumn("blacklisted_category", F.regexp_extract("path", r'([^/]+)/domains$', 1)).drop("value", "path")
blacklisted_urls_df.show()

banned_words_df = spark.read.text("/content/data/banned_words.txt")
banned_words_df.show()

+--------------------+--------------------+
|     blacklisted_url|blacklisted_category|
+--------------------+--------------------+
|        0-1avsex.com|               adult|
|          0-1sex.com|               adult|
|     0-800-email.com|               adult|
| 0-assist-protect.cf|               adult|
| 0-assist-protect.ga|               adult|
|           0-dix.com|               adult|
|  0-livechatlady.com|               adult|
|         0-porno.com|               adult|
|0-puppy-0.tumblr.com|               adult|
| 0-secure-paypal.com|               adult|
|          0-shop.com|               adult|
|0-start-trizor.gi...|               adult|
| 0-syxela.tumblr.com|               adult|
|0-transsexuel-bre...|               adult|
|0.0.0.0.0.0.0.0.0...|               adult|
|0.0.0.0.0.0.0.0.0...|               adult|
|0.0.0.0.0.0.0.0.c...|               adult|
|0.0.0.0.0.rencont...|               adult|
|  0.0.0.0.09.free.fr|               adult|
|   0.0.0.005.free.fr|          

---
---
## Your Code Edits Start from Here


### Common Imports and Functions

List imports and functions that are used across different questions in these sections.

In [12]:
#######################################
###!@0.1 START COMMON USER IMPORTS
#######################################
## Specify valid imports, if any, for ALL your answers  ==========
## start your edits here =================

import re
import unicodedata
from transformers import GPT2Tokenizer
from pyspark.sql.functions import col, lower, udf, size, split, monotonically_increasing_id, explode
from pyspark.sql.functions import length, asc, desc, row_number
from pyspark.sql.types import BooleanType, StringType, ArrayType, IntegerType
from pyspark.sql.window import Window

## end your edits here =================
###!@0.1 END COMMON USER IMPORTS

In [13]:
#######################################
###!@0.2 START COMMON USER FUNCTIONS
#######################################
## Specify user defined functions, if any, used by multiple answers   =====
## start your edits here =================

def build_regex(substring):
  substring = substring.lower()
  return ".*".join(map(re.escape, substring))

def clean_text(text):
  if text is None: return None

  # Make content lowercase
  text = text.lower()

  # Replace all digits with 0
  text = re.sub(r"\d", "0", text)

  # Normalize accent characters
  text = unicodedata.normalize("NFKD", text)
  text = "".join(c for c in text if not unicodedata.combining(c))

  # Remove unwanted characters
  text = re.sub(r"[^\w\s\n\r]", "", text)

  # Normalize tabs and multiple spaces
  text = re.sub(r"\s+", " ", text)

  return text.strip()

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
def extract_unique_ngrams(text, n=5):
  if text is None: return []
  tokens = tokenizer.encode(text)
  ngrams = zip(*[tokens[i:] for i in range(n)])
  unique_ngrams = {tuple(ng) for ng in ngrams}
  return [list(ng) for ng in unique_ngrams]

# Use for question 3.2
def get_minhash_signature(ngrams):
  return [min(hash_ngram(ngram, i) for ngram in ngrams) for i in range(NUM_HASHES)]

def get_buckets(signature, r=8, b=3):
  buckets = [tuple(signature[i * b:(i + 1) * b]) for i in range(r)]
  buckets = [tuple(str(x) for x in bucket) for bucket in buckets]
  # Covert to strings
  buckets = ["-".join(bucket) for bucket in buckets]
  return buckets

## end your edits here =================
###!@0.2 END COMMON USER FUNCTIONS

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

### Question 0: Filter and Sort only English pages ***(5 points)***
----

You are interested only in looking at the pages which use the English Language, as our model is English Language based. Filter out all pages having any other languages and store the data in a new dataframe.

**Output:** A dataframe with same columns as in the input dataframe, but retaining only the rows which have content in English language (`eng`).

**SORTING FOR VALIDATION:** Sort on `url` column (lexical order).

In [14]:
#######################################
###!@0.3 START ANSWER SET 0

### Q0.3 ###################################################

def question_0(input_df):

  ## start your edits here =================

  output_df = input_df.filter(input_df["language"] == "eng")

  ## end your edits here =================

  return output_df

###!@0.3 END ANSWER SET 0

## Question 1: URL Filtering
 ------

 URL filtering is a crucial preprocessing step before training large language models (LLMs) to ensure the quality, relevance, and safety of the training data. Since datasets like Common Crawl contain vast amounts of web data, not all sources are suitable for model training. Filtering helps remove low-quality, harmful, or biased content, preventing the model from learning misinformation, toxic language, or irrelevant data. It also enhances efficiency by reducing the dataset size, focusing on high-quality sources, and mitigating legal and ethical risks. Proper URL filtering leads to better generalization, improved fairness, and a more reliable AI system.

### Question 1.1: Remove URLs based on blacklisted terms ***(10 points)***

- We need to remove all URLs from the input dataframe which fall under specific categories that are provided to you in the list `blacklist_terms`, e.g., ["cryptojacking", "gambling", "stalkerware", "mixed_adult"]. The category for each such blacklisted URL is given in `blacklisted_urls_df`.

**Output:** A dataframe with specific rows filtered out as per this rule.

**SORTING FOR VALIDATION:** Sort on `url` column (lexical order).

In [15]:
#######################################
###!@1.1 START ANSWER SET 1.1

### Q1.1 ###################################################

def question_1_1(input_df, blacklisted_urls_df, blacklist_terms):

  ## start your edits here  =================

  remove_df = blacklisted_urls_df.filter(blacklisted_urls_df["blacklisted_category"].isin(blacklist_terms))
  results_df = input_df.join(remove_df, input_df["url"] == remove_df["blacklisted_url"], "left_anti")

  ## end your edits here =================

  return results_df

###!@1.1 END ANSWER SET 1.1

### Question 1.2: Remove URLs based on Banned Words/Phrases ***(15 points)***

Remove all rows with URLs containing banned words/phrases within their URL string. The list of banned words are provided in the input dataframe `banned_words_df`.

- Hard whole word matching. Any URL containing the entire banned word contiguously. e.g., the URLs matching the banned word `badword` include `badwords.com`, `verybadwordpage.org`, `foo.com/mybadwords`, etc.
- Strict sub-word matching. Any number of non-alphanumeric characters between any of characters in the banned word. e.g., the URLs matching the banned word `badword` include `bad.word`, `badw?ord`, `bad.w.ord`, `bad%20%20word`, etc.

Whitespace character " " is replaced by "%20" in the URL.
The matching should be case-insensitive.

**Output:** A dataframe with specific rows filtered out as per this rule.

**SORTING FOR VALIDATION:** Sort on `url` column (lexical order).

In [16]:
#######################################
###!@1.2 START ANSWER SET 1.2

### Q1.2 ###################################################

def question_1_2(input_df, banned_words_df):

  ## start your edits here  =================

  pattern_list = [row["value"] for row in banned_words_df.select("value").collect()]
  compiled_patterns = [build_regex(p) for p in pattern_list]
  compiled_pattern = "|".join(compiled_patterns)
  filtered_df = input_df.filter(~lower(col("url")).rlike(compiled_pattern))

  ## end your edits here =================

  return filtered_df

###!@1.2 END ANSWER SET 1.2

## Question 2: Page Content Cleanup ***(20 points)***

-----

Raw web data often contains inconsistencies such as excessive whitespace, unwanted symbols, HTML tags, special characters, and encoding errors, which can introduce noise and degrade model performance. Techniques like whitespace normalization, symbol removal, stopword filtering, and text deduplication help enhance data consistency, improve tokenization efficiency, and prevent the model from learning redundant or irrelevant patterns. Proper preprocessing not only optimizes training efficiency but also improves downstream tasks like text generation, retrieval, and reasoning, leading to more reliable and coherent AI systems.

Do the following on the `content` of the webpages:
- Convert all text to lower case, e.g., `AbcDE` with `abcde`.
- Replace all digits from 0-9 with a placeholder "0", e.g., `12345` with `00000`
- Modify any letter with an accent to the base letter using the `unicode` Python package, e.g., `é` with `e`.
- Remove anything that is not a letter, number or a whitespace character (spaces, tabs, newlines, etc) from the text, e.g., `ab._ c- d` with `ab c d`
- Normalize tabs with spaces, and contiguous spaces with single space. characters and by replacing them with a single space, " ". (Do not replace/remove `\n`, `\r` characters).
- Only retain rows with content whose length is greater than or equal to 5 words, after performing the above operations, e.g., content value `lorem ipsum dolor sit` should be removed since it has only 4 words whereas content with value `lorem ipsum dolor sit amet` should be kept.

**Output:** Return a dataframe with two columns named `url` and `normalized_content` that has applied the above cleanup and removed those rows with fewer than 5 words in length.

**SORTING FOR VALIDATION:** Sort on `url` column (lexical order).



In [17]:
#######################################
###!@2 START ANSWER SET 2

### Q2 ###################################################

def question_2(input_df):

  ## start your edits here  =================

  clean_text_udf = udf(clean_text, StringType())
  whitespaced_removed_df = input_df.withColumn("normalized_content", clean_text_udf(col("content")))
  whitespaced_removed_df = whitespaced_removed_df.filter(size(split(col("content"), " ")) >= 5)
  whitespaced_removed_df = whitespaced_removed_df.select("url", "normalized_content")

  ## end your edits here =================

  return whitespaced_removed_df


###!@2 END ANSWER SET 2

## Question 3: Content Deduplication

-----

Deduplication is a crucial preprocessing step before training LLMs to prevent redundancy and improve training efficiency. Large-scale web datasets often contain duplicate or near-duplicate text from scraped web pages, social media posts, and boilerplate content. Training on redundant data leads to inefficient use of computational resources, overfitting on frequently repeated content, and potential biases in model outputs. By removing exact and near-duplicate texts, deduplication enhances dataset diversity, ensures better generalization, and reduces the risk of memorization, which is essential for ethical AI deployment. A well-deduplicated dataset results in more efficient training and a more robust, diverse, and informative language model.

### Question 3.1:  Finding unique 5-grams ***(15 points)***

The GPT-2 tokenizer is a Byte Pair Encoding (BPE) tokenizer used by OpenAI's GPT-2 model. It tokenizes text into subword units rather than full words, which helps the model handle rare words and different languages more efficiently. Each word (token) is replaced by an integer. More details here: https://huggingface.co/docs/transformers/en/model_doc/gpt2.

This question will generate all unique 5-grams (sets of 5 consecutive words) in a given text dataset. The 5-grams  internally are order-sensitive, i.e., `[1,2,3,4,5]` is different from `[1,4,3,2,5]`.

Tokenize the `normalized_content` column in the input dataframe using the GPT2 tokenizer and obtain the set of unique 5-grams for each row. Each 5-gram is a `list[int]` with 5 integers. Place all the 5-gram lists for a URL in a single column called `5grams_unique` and return a dataframe having `URL` and `5grams_unique` columns. The `5grams_unique` column will contain an `list[list[int]]`.

**OUTPUT:** Dataframe with `URL`, `5grams_unique` and `normalized_content` columns, with the latter having all 5-grams for the content corresponding to that URL in input dataframe.

**SORTING FOR VALIDATION:** Sort on `url` column (lexical order), and then sort the unique `5-grams` in numeric order (`[1,2,3,4,5],[1,2,3,4,6],[2,3,4,5,1],...`).

In [18]:
#######################################
###!@3.1 START ANSWER SET 3.1

### Q3.1 ###################################################


def question_3_1(input_df):

  ## start your edits here  =================

  extract_ngrams = udf(lambda x: extract_unique_ngrams(x, n=5), ArrayType(ArrayType(IntegerType())))

  unique_ngrams_df = input_df.withColumn("5grams_unique", extract_ngrams(input_df["normalized_content"]))

  ## end your edits here =================

  return unique_ngrams_df

###!@3.1 END ANSWER SET 3.1

### Question 3.2:  Page Level Deduplication ***(30 points)***


The goal of this question is to filter out pages from the vast Common Crawl data that are duplicates, nearly duplicates or similar in content to each other and retain only one of all the found duplicates to ensure no bias in the LLM training.

We need to generate MinHash signatures for the unique 5-grams for the text content, found in Question 3.1. MinHash is commonly used in Locality-Sensitive Hashing (LSH).

The input to this is a dataframe with three columns, `url`, `list[list[int]]` having the unique 5-grams for the document and the `normalized_content` column.
The *document signature* consists of the 24 minhashes over these 5-grams.
To obtain the `i`th minhash for a document, find the `i`th hash for all the 5-grams in a document by calling `hash_ngram` (pass each of the the n-gram and `i` as arguments to this function) to return the `i`th hash for it as an integer. Then find the minimum of the n-gram hash values forms the `i`th minhash for that document. Similarly, obtain all 25 minhashes for each document with `i` ranging from 0 to 23.

An *ordered list* (0th minhash to 23th minhash) of these 24 minhashes is the document's **signature**.

This document signature needs to be split into `r=8` buckets, with `b=3` minhash numbers each, while maintaining their order internally, i.e., minhash `0-2` for a document is placed in bucket `1`, `3-5` in bucket `2`, and so on.

For deduplication, cluster the documents such that if all minhases in *at least one bucket* of two documents are exactly same, they belong to the same cluster. For example, if document A has minhashes [1,2,3] in its 3rd bucket and document B also has the same minhashes [1,2,3] in that bucket, then they belong to the same cluster. Similarly, if B and C share the same minhashes in bucket 7, they belong to the same cluster. We follow the *commutative property*: if A and B are in the same cluster, and B and C are in the same cluster, then A, B, C belong to the same cluster.

Now, from each cluster, drop all but the longest document. If two documents have the same length, retain the one that is lexically first. To find the length of the document, use the number of words in the `normalized_content` column for that document.

**OUTPUT:** Dataframe with `URL` and `normalized_content` columns, with the latter having the content corresponding to that URL in input dataframe.

**SORTING FOR VALIDATION:** Sort on `url` column (lexical order)

In [19]:
#######################################
###!@3.2 START ANSWER SET 3.2

NUM_HASHES = 24  # Number of hash functions
M = 2**32 - 1  # Large prime number for modulus

def hash_ngram(ngram, i):
  ngram = list(ngram)
  ngram.append(i)
  return hash(tuple(ngram)) % M

### Q3.2 ###################################################


def question_3_2(unique_ngrams_df):

  ## start your edits here  =================

  # Extract Signatures
  extract_signature = udf(lambda ngrams: get_minhash_signature(ngrams), ArrayType(IntegerType()))
  unique_ngrams_df = unique_ngrams_df.withColumn("signature", extract_signature(unique_ngrams_df["5grams_unique"]))

  # Make buckets from signatures
  extract_buckets = udf(lambda signs: get_buckets(signs), ArrayType(StringType()))
  unique_ngrams_df = unique_ngrams_df.withColumn("buckets", extract_buckets(unique_ngrams_df["signature"]))

  # Match documents with same bucket
  unique_ngrams_df = unique_ngrams_df.withColumn("doc_id", monotonically_increasing_id())
  unique_ngrams_df_explode = unique_ngrams_df.select("doc_id", explode("buckets").alias("bucket_key"))

  doc_links = unique_ngrams_df_explode.alias("a").join(
                unique_ngrams_df_explode.alias("b"),
                on="bucket_key"
            ).where(col("a.doc_id") < col("b.doc_id")) \
            .select(col("a.doc_id").alias("doc1"), col("b.doc_id").alias("doc2"))

  labels = unique_ngrams_df.select("doc_id").distinct().withColumnRenamed("doc_id", "id") \
           .withColumn("label", col("id"))

  changed = True

  while changed:
      # Join to propagate labels from doc1 to doc2
      new_labels = doc_links.join(labels, doc_links.doc1 == labels.id) \
                            .select(col("doc2").alias("id"), col("label"))

      # Combine old and new labels, always take the min
      updated_labels = labels.union(new_labels) \
                            .groupBy("id").agg({"label": "min"}) \
                            .withColumnRenamed("min(label)", "label")

      # Check if labels changed
      changed = updated_labels.join(labels, "id").filter(labels.label != updated_labels.label).count() > 0

      # Update
      labels = updated_labels

  output_df = unique_ngrams_df.join(labels, unique_ngrams_df.doc_id == labels.id, "left").drop("id")

  # Rank the documents based on content length and take the ones with largest
  output_df = output_df.withColumn("content_length", length("normalized_content"))

  window = Window.partitionBy("label") \
               .orderBy(desc("content_length"), asc("normalized_content"))

  output_df = output_df.withColumn("rank", row_number().over(window))

  output_df = output_df.filter(col("rank") == 1).drop("rank","content_length","signature","buckets","doc_id","label")

  ## end your edits here =================

  return output_df

###!@3.2 END ANSWER SET 3.2

## Question 4: Load Balancing of Unique n-grams across partitions ***(20 points)***

-----

Balancing the distribution of n-grams across partitions is crucial to ensure that workloads are evenly distributed among computing resources. Load imbalance—where some partitions process significantly more n-grams than others—can lead to bottlenecks, inefficient resource utilization, and prolonged preprocessing times.

An even distribution of n-grams across partitions is essential for parallel processing efficiency, preventing stragglers that delay the entire pipeline. Additionally, it ensures that each compute node handles a fair share of the workload, maximizing GPU/CPU utilization and minimizing memory contention. Proper load balancing also plays a crucial role in maintaining data consistency, particularly in tasks such as vocabulary extraction, frequency-based filtering, and tokenization, which are fundamental for optimizing model quality.

**NOTE: You MUST use RDDs for this question ONLY.**

The goal of this question is to observe load imbalancing of unique 5-grams you generated from Question 3.1. across RDD partitions created under the hood from the Spark Dataframe and measure its impact.

First, check the number of partitions that the dataframe is stored in. Perform the "task" described below.

Now, repartition the dataframe, but DO NOT change the number of partitions, and run the same "task".

**Task:** "Explode" the dataframe using the `5grams_unique` column such that each n-gram is now stored in a new row. Now, count all distinct n-grams across all URLs.

We will call you function twice, with `bool repart` set to `True` or `False`. If false, then you will not repartition and return the answer. If true, then you WILL repartition and return the answer. We will measure the time taken for each of these function calls.

**OUTPUT:** Dataframe with the list of unique 5-grams, `each5gram_unique` column, for each invocation.

**SORTING FOR VALIDATION:** `each5gram_unique` in numeric order.






In [28]:
#######################################
###!@4 START ANSWER SET 4

### Q4 ###################################################


def question_4(unique_ngrams_df, repart):

  ## end your edits here =================
  num_partitions = unique_ngrams_df.rdd.getNumPartitions()
  # Run it only when required
  if repart and num_partitions > 1:
    unique_ngrams_df = unique_ngrams_df.withColumn("num_5grams", size(col("5grams_unique")))
    # Calculate number of 5grams for each partition
    total_5grams = unique_ngrams_df.selectExpr("sum(num_5grams)").collect()[0][0]
    grams_per_partition = total_5grams / num_partitions
    # Partition the data manually
    df_indexed = unique_ngrams_df.withColumn("row_id", monotonically_increasing_id())
    row_data = df_indexed.select("row_id", "num_5grams").orderBy("row_id").collect()
    partition_assignments = []
    current_partition = 0
    current_sum = 0

    # Assign the partitions
    for row in row_data:
        row_id = row["row_id"]
        count = row["num_5grams"]
        if current_sum + count > grams_per_partition and current_partition < num_partitions - 1:
            current_partition += 1
            current_sum = 0
        partition_assignments.append((row_id, current_partition))
        current_sum += count

    # Partition the data
    assignment_df = spark.createDataFrame(partition_assignments, ["row_id", "partition_id"])
    df_partitioned = df_indexed.join(assignment_df, on="row_id", how="left") \
                                .drop("row_id", "num_5grams")
    unique_ngrams_df = df_partitioned.repartition(num_partitions, "partition_id").drop("partition_id")

  # Perform explosion
  rdd = unique_ngrams_df.select("5grams_unique").rdd
  exploded_rdd = rdd.flatMap(lambda row: row[0])
  unique_5grams = exploded_rdd.map(lambda x: tuple(x)).distinct()
  unique_5grams = unique_5grams.map(lambda x: list(x))
  output_df = unique_5grams.map(lambda x: (x, )).toDF(["each5gram_unique"])

  ## start your edits here  =================

  return output_df

###!@4 END ANSWER SET 4

### Evaluation Section

In this section, each output dataframe from the previous question is passed to the next question's function as an input. So, ensure that you DO NOT MODIFY the arguments being passed to the function for each question and must solve them in the given order.

In [29]:
###!@5 START ANSWER SET EVALUATION
# ========== *** DO NOT MODIFY *** ========== #

ans0_output_df = question_0(input_df)
ans0_output_df.show()
ans0_output_df.write.mode("overwrite").parquet("ans0_output_df.parquet")

blacklist_terms = ["cryptojacking", "gambling", "stalkerware", "mixed_adult"]
ans1_1_output_df = question_1_1(ans0_output_df, blacklisted_urls_df, blacklist_terms)
ans1_1_output_df.show()
ans1_1_output_df.write.mode("overwrite").parquet("ans1_1_output_df.parquet")

ans1_2_output_df = question_1_2(ans1_1_output_df, banned_words_df)
ans1_2_output_df.show()
ans1_2_output_df.write.mode("overwrite").parquet("ans1_2_output_df.parquet")

ans2_output_df = question_2(ans1_2_output_df)
ans2_output_df.show()
ans2_output_df.write.mode("overwrite").parquet("ans2_output_df.parquet")

ans3_1_output_df = question_3_1(ans2_output_df)
ans3_1_output_df.show()
ans3_1_output_df.write.mode("overwrite").parquet("ans3_1_output_df.parquet")

ans3_2_output_df = question_3_2(ans3_1_output_df)
ans3_2_output_df.count()
ans3_2_output_df.write.mode("overwrite").parquet("ans3_2_output_df.parquet")

ans4a_output_df = question_4(ans3_2_output_df, False)
ans4a_output_df.count()
ans4a_output_df.write.mode("overwrite").parquet("ans4a_output_df.parquet")

ans4b_output_df = question_4(ans3_2_output_df, True)
ans4b_output_df.count()
ans4b_output_df.write.mode("overwrite").parquet("ans4b_output_df.parquet")

# ========== *** DO NOT MODIFY *** ========== #
###!@5 END ANSWER SET EVALUATION